# SPANDAN · 02 · IMS run-to-failure health indicators (input for RUL, tasks M2-24 / M2-25)

Turns every 1-second **NASA IMS** snapshot into health indicators per bearing (RMS, kurtosis, physics
rule scores at 3.2 kHz), one CSV per test. The RUL model is fitted on these CSVs (next task).

**Before running**
1. *Add data*: attach `vinayak123tyagi/bearing-dataset` (folders `1st_test`, `2nd_test`, `3rd_test`).
2. *Settings*: **Internet on**. CPU is enough.
3. *Run all*. The full data (~9,500 snapshots) takes roughly 15–30 minutes; set `EVERY = 3` for a quick run.

**After running**: download `/kaggle/working/results/ims_test*_indicators.csv` into `ml/results/`.

In [ ]:
# Get the SPANDAN code (public repo). Needs "Internet on" in the notebook settings.
import os, re, sys, subprocess
REPO = "/kaggle/working/SPANDAN"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)  # rerun in the same session: get the latest code
else:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Duggineniakhil/SPANDAN.git", REPO], check=True)
sys.path.insert(0, f"{REPO}/ml")

# Install the exact scikit-learn version the backend uses, so the saved model loads there.
# Must run before anything imports sklearn (if it was imported already: Run > Restart & clear outputs, then Run all).
pin = re.search(r"^scikit-learn==([\d.]+)", open(f"{REPO}/ml/requirements.txt").read(), re.M).group(1)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={pin}"], check=True)
import numpy, scipy, sklearn
print("numpy", numpy.__version__, "| scipy", scipy.__version__, "| scikit-learn", sklearn.__version__)
assert sklearn.__version__ == pin, f"scikit-learn {sklearn.__version__} is loaded, need {pin}: restart the kernel and run all"


In [ ]:
from spandan_ml import ims
sets = ims.discover("/kaggle/input")
for s in sets: print(f"test {s.number}: {len(s.snapshots)} snapshots in {s.folder}")
if not sets:
    # Some mirrors ship the original .rar/.7z archives: show what is there.
    import subprocess; print(subprocess.run(["bash", "-lc", "find /kaggle/input -maxdepth 4 | head -50"], capture_output=True, text=True).stdout)
    raise SystemExit("No IMS snapshot folders found (see listing above)")

In [ ]:
EVERY = 1  # use every n-th snapshot (3 = quick run)
from spandan_ml.pipelines import run_ims
paths = run_ims("/kaggle/input", "/kaggle/working/results", every=EVERY)
paths

In [ ]:
import csv
import matplotlib.pyplot as plt
for p in paths:
    rows = list(csv.DictReader(open(p)))
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    for b in "1234":
        rb = [r for r in rows if r["bearing"] == b]
        label = f"bearing {b}" + (f" ({rb[0]['failed_with']})" if rb and rb[0]["failed_with"] else "")
        axes[0].plot([float(r["rms_raw"]) for r in rb], label=label)
        axes[1].plot([float(r["kurtosis_raw"]) for r in rb], label=label)
    axes[0].set_title(f"{p.name}: RMS"); axes[1].set_title("kurtosis"); axes[0].legend(fontsize=7)
    plt.show()